# FOLD-TOMO-01: Trace-Sufficient Reverse Fold Engine

## Nexus Fold Tomography Notebook

**Umbrella name:** `Nexus Fold Tomography`  
**Short code:** `FOLD-TOMO`  
**Notebook:** `FOLD-TOMO-01`  
**Engine:** `Trace-Sufficient Reverse Fold Engine` (`TSRFE`)  
**Primary pin:** `448 Nyquist Pin`  
**Primary theorem:** `Dyadic Terminal Tomography Theorem` (`DTT`)  

$$
\boxed{
\text{Forward collapse hides address; reverse recovery restores address by reading shape.}
}
$$

This notebook verifies the current fold lock and turns it into a working proof engine.

Core pipeline:

$$
d_i^{(\ell+1)}=\left|d_{i+1}^{(\ell)}-d_i^{(\ell)}\right|
$$

$$
x_i^{(\ell)}=d_i^{(\ell)}\bmod2
$$

$$
x_i^{(\ell+1)}=x_i^{(\ell)}\oplus x_{i+1}^{(\ell)}
$$

$$
x^{(\ell)}=(I+E)^\ell x^{(0)}
$$

$$
x_i^{(\ell)}
=
\bigoplus_{j\subseteq\ell}x_{i+j}^{(0)}
$$


# 0. Naming Lock

Use this naming stack so the work does not fragment across sessions or AI systems.

| Object | Name | Short code |
|---|---|---|
| Research grouping | Nexus Fold Tomography | `FOLD-TOMO` |
| Reverse engine | Trace-Sufficient Reverse Fold Engine | `TSRFE` |
| Main exact system | Rule-90 parity shadow of decimal fold | `Parity Shadow` |
| Main structural pin | Level 448 eight-point 64-grid probe | `448 Nyquist Pin` |
| Terminal theorem | Dyadic terminal rows are residue-class parity checks | `DTT` |
| Signature object | Fold fingerprint | `F(D,F)` |
| Final thesis | Shape is memory; location is the missing variable | `Shape Memory Lock` |


In [ ]:
# Optional dependency setup.
import sys, subprocess, importlib.util

required = ["numpy", "matplotlib", "mpmath"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")


In [ ]:
import math
import random
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True


# 1. Seed: First 2048 Fractional Digits of $\pi$

The verified seed is the first $2048$ digits **after the decimal point** of $\pi$.

This convention gives:

$$
S_0^{(10)}=9338.
$$


In [ ]:
def pi_fractional_digits(n=2048, guard=50):
    # Return the first n fractional decimal digits of pi as a string.
    mp.mp.dps = n + guard
    s = str(mp.pi)
    frac = s.split(".")[1]
    return frac[:n]

N = 2048
digits = pi_fractional_digits(N)

assert len(digits) == N
assert digits[:20] == "14159265358979323846"

decimal_seed = [int(c) for c in digits]
raw_decimal_sum = sum(decimal_seed)

print("N:", N)
print("First 32 fractional digits of pi:", digits[:32])
print("Raw decimal sum S_0^(10):", raw_decimal_sum)

assert raw_decimal_sum == 9338


# 2. Decimal Fold and Parity Shadow

The parity shadow is exact because:

$$
|a-b|\bmod2=(a+b)\bmod2=a\oplus b.
$$


In [ ]:
def decimal_difference_rows(seed):
    rows = [list(seed)]
    cur = list(seed)
    while len(cur) > 1:
        cur = [abs(b - a) for a, b in zip(cur, cur[1:])]
        rows.append(cur)
    return rows

def xor_rows(seed_bits):
    rows = [list(seed_bits)]
    cur = list(seed_bits)
    while len(cur) > 1:
        cur = [a ^ b for a, b in zip(cur, cur[1:])]
        rows.append(cur)
    return rows

parity_seed = [d & 1 for d in decimal_seed]

decimal_rows = decimal_difference_rows(decimal_seed)
xor_lattice = xor_rows(parity_seed)

parity_shadow_ok = all(
    [(v & 1) for v in decimal_rows[level]] == xor_lattice[level]
    for level in range(N)
)

print("Parity shadow exact across all levels:", parity_shadow_ok)
assert parity_shadow_ok


# 3. Glyph Reader Trace Metrics

$$
N_\ell=\text{row length}
$$

$$
S_\ell=\sum_i x_i^{(\ell)}
$$

$$
\rho_\ell=\frac{S_\ell}{N_\ell}
$$

$$
R_\ell=S_\ell-\frac{N_\ell}{2}
$$

$$
R_\ell=0\quad\Longleftrightarrow\quad\text{exact half-density lock}
$$


In [ ]:
def trace_metrics(rows):
    N_vals = np.array([len(r) for r in rows], dtype=int)
    S_vals = np.array([sum(r) for r in rows], dtype=int)
    rho = S_vals / N_vals
    R = S_vals - N_vals / 2.0
    dS = np.diff(S_vals)
    dR = np.diff(R)
    return {
        "N": N_vals,
        "S": S_vals,
        "rho": rho,
        "R": R,
        "dS": dS,
        "dR": dR,
        "locks": [i for i, val in enumerate(R) if val == 0],
    }

binary_trace = trace_metrics(xor_lattice)

decimal_N = np.array([len(r) for r in decimal_rows], dtype=int)
decimal_S = np.array([sum(r) for r in decimal_rows], dtype=int)
decimal_rho = decimal_S / decimal_N

print("Binary parity seed:")
print("  N_0 =", binary_trace["N"][0])
print("  S_0 =", binary_trace["S"][0])
print("  rho_0 =", binary_trace["rho"][0])
print("  R_0 =", binary_trace["R"][0])
print()
print("Decimal value channel:")
print("  S_0^(10) =", decimal_S[0])
print("  S_13^(10) =", decimal_S[13])
print("  Amplitude collapse by level 13:", (decimal_S[0] - decimal_S[13]) / decimal_S[0])
print()
print("Half-density locks:", len(binary_trace["locks"]))
print("First 15 locks:", binary_trace["locks"][:15])
print("Last 10 locks:", binary_trace["locks"][-10:])


# 4. Verified Lock Points

Core verified points:

$$
N_{448}=1600,\quad S_{448}=800,\quad R_{448}=0.
$$

$$
N_{512}=1536,\quad S_{512}=764,\quad R_{512}=-4.
$$

Terminal:

$$
x^{(2046)}=[1,1],
\qquad
x^{(2047)}=[0].
$$


In [ ]:
def show_level(level):
    row = xor_lattice[level]
    Nl = len(row)
    Sl = sum(row)
    Rl = Sl - Nl / 2
    print(f"Level {level}: N={Nl}, S={Sl}, rho={Sl/Nl:.6f}, R={Rl}")
    if Nl <= 32:
        print(" row:", row)

for level in [0, 13, 110, 448, 512, 2032, 2040, 2044, 2046, 2047]:
    show_level(level)

assert binary_trace["N"][448] == 1600
assert binary_trace["S"][448] == 800
assert binary_trace["R"][448] == 0

assert binary_trace["N"][512] == 1536
assert binary_trace["S"][512] == 764
assert binary_trace["R"][512] == -4

assert xor_lattice[2046] == [1, 1]
assert xor_lattice[2047] == [0]


# 5. Lucas Masks

The closed form is:

$$
x_i^{(\ell)}
=
\bigoplus_{j\subseteq\ell}x_{i+j}^{(0)}.
$$

For $\ell=448$:

$$
448=256+128+64
$$

so:

$$
j\in\{0,64,128,192,256,320,384,448\}.
$$

For $\ell=512$:

$$
j\in\{0,512\}.
$$


In [ ]:
def lucas_offsets(level):
    # Return all offsets j such that j is a bit-subset of level.
    bits = [1 << k for k in range(level.bit_length()) if (level >> k) & 1]
    offsets = [0]
    for bit in bits:
        offsets += [o + bit for o in offsets]
    return sorted(offsets)

for level in [110, 448, 512, 640, 2040, 2044, 2046, 2047]:
    offsets = lucas_offsets(level)
    print(f"level={level:4d}, popcount={level.bit_count():2d}, mask_size={len(offsets):4d}")
    print(" offsets:", offsets[:16], "..." if len(offsets) > 16 else "")
    print()

assert lucas_offsets(448) == [0,64,128,192,256,320,384,448]
assert lucas_offsets(512) == [0,512]


In [ ]:
def row_from_lucas(seed_bits, level):
    offsets = lucas_offsets(level)
    row_len = len(seed_bits) - level
    out = []
    for i in range(row_len):
        v = 0
        for j in offsets:
            v ^= seed_bits[i + j]
        out.append(v)
    return out

for level in [0, 1, 2, 3, 13, 110, 448, 512, 640, 1024, 2040, 2044, 2046, 2047]:
    direct = row_from_lucas(parity_seed, level)
    assert direct == xor_lattice[level], f"Lucas direct row mismatch at level {level}"

print("Lucas direct computation matches recursive XOR lattice at all tested levels.")


# 6. Dyadic Terminal Tomography Theorem

For:

$$
N=2^m
$$

and:

$$
\ell_k=N-2^k,
$$

the row is:

$$
x_i^{(N-2^k)}
=
\bigoplus_{q=0}^{2^{m-k}-1}
x_{i+q2^k}^{(0)}.
$$

Each terminal row is a residue-class checksum modulo $2^k$.


In [ ]:
def dyadic_terminal_row_from_seed(seed_bits, k):
    N = len(seed_bits)
    modulus = 1 << k
    out = []
    for i in range(modulus):
        v = 0
        for q in range(i, N, modulus):
            v ^= seed_bits[q]
        out.append(v)
    return out

m = int(math.log2(N))
assert 2**m == N

for k in range(0, m):
    level = N - (1 << k)
    theorem_row = dyadic_terminal_row_from_seed(parity_seed, k)
    lattice_row = xor_lattice[level]
    ok = theorem_row == lattice_row
    print(f"k={k:2d}, level={level:4d}, row_len={len(lattice_row):4d}, theorem_match={ok}")
    if k <= 4:
        print(" row:", lattice_row)
    assert ok


# 7. GF(2) Rank Engine With Integer Bitmasks

A parity equation can be represented as one Python integer bitmask.

The coefficient row has a `1` wherever the original seed bit participates in the XOR.


In [ ]:
def masks_for_level(N, level):
    offsets = lucas_offsets(level)
    masks = []
    for i in range(N - level):
        mask = 0
        for j in offsets:
            mask |= 1 << (i + j)
        masks.append(mask)
    return masks

def gf2_rank_int(rows):
    # Rank of integer bitmask rows over GF(2).
    basis = {}
    for row in rows:
        x = int(row)
        while x:
            pivot = x.bit_length() - 1
            if pivot in basis:
                x ^= basis[pivot]
            else:
                basis[pivot] = x
                break
    return len(basis)

dyadic_masks = []
for k in range(0, m):
    level = N - (1 << k)
    dyadic_masks.extend(masks_for_level(N, level))

masks_448 = masks_for_level(N, 448)
masks_512 = masks_for_level(N, 512)

rank_dyadic = gf2_rank_int(dyadic_masks)
rank_448 = gf2_rank_int(masks_448)
rank_dyadic_448 = gf2_rank_int(dyadic_masks + masks_448)
rank_dyadic_512 = gf2_rank_int(dyadic_masks + masks_512)
rank_all = gf2_rank_int(dyadic_masks + masks_448 + masks_512)

print("Number of dyadic terminal rows:", len(dyadic_masks))
print("rank(dyadic) =", rank_dyadic)
print("rank(448) =", rank_448)
print("rank(dyadic + 448) =", rank_dyadic_448)
print("rank(dyadic + 512) =", rank_dyadic_512)
print("rank(dyadic + 448 + 512) =", rank_all)
print("Remaining degrees after dyadic + 448:", N - rank_dyadic_448)

assert rank_dyadic == 1024
assert rank_dyadic_448 == 1600
assert N - rank_dyadic_448 == 448


# 8. Reverse Entropy of the 448 Pin

For:

$$
448=256+128+64,
$$

the factorization is:

$$
(I+E)^{448}
=
(I+E^{256})(I+E^{128})(I+E^{64}).
$$

Each factor $I+E^s$ is reversed by choosing $s$ boundary bits and propagating:

$$
x_{i+s}=x_i\oplus y_i.
$$

Total boundary entropy:

$$
256+128+64=448.
$$


In [ ]:
def reverse_one_shift_xor(y, shift, boundary):
    # Reverse y_i = x_i XOR x_{i+shift}.
    assert len(boundary) == shift
    x = list(boundary) + [0] * len(y)
    for i, yi in enumerate(y):
        x[i + shift] = x[i] ^ yi
    return x

x_true = [1,0,1,1,0,0,1,0]
shift = 2
y = [x_true[i] ^ x_true[i+shift] for i in range(len(x_true)-shift)]
x_rec = reverse_one_shift_xor(y, shift=shift, boundary=x_true[:shift])

print("x_true:", x_true)
print("y:", y)
print("boundary:", x_true[:shift])
print("x_rec:", x_rec)
assert x_rec == x_true


# 9. Weight Constraints

Full rows give linear constraints:

$$
A_\ell x=y_\ell.
$$

Row sums give nonlinear Hamming-weight constraints:

$$
\operatorname{wt}(A_\ell x)=S_\ell.
$$

These are the constraints that break residual symmetry and filter the affine space.


In [ ]:
nonzero_R_levels = [i for i, R in enumerate(binary_trace["R"]) if R != 0]
zero_R_levels = binary_trace["locks"]

print("R=0 levels:", len(zero_R_levels))
print("R!=0 levels:", len(nonzero_R_levels))
print("First 20 R=0 levels:", zero_R_levels[:20])
print("Last 20 R=0 levels:", zero_R_levels[-20:])

assert len(nonzero_R_levels) == 1929


# 10. Small-N Proof Engine: Weight-Trace Rigidity

For small $N$, enumerate all seeds and group them by the weight trace:

$$
\mathcal W(x)
=
\left(
\operatorname{wt}((I+E)^\ell x)
\right)_{\ell=0}^{N-1}.
$$

This measures whether row-sum shape traces store address information.


In [ ]:
def bits_from_int(n, width):
    return [(n >> i) & 1 for i in range(width)]

def weight_trace(bits):
    rows = xor_rows(bits)
    return tuple(sum(r) for r in rows)

def enumerate_weight_trace_classes(width=16):
    classes = defaultdict(list)
    for n in range(1 << width):
        bits = bits_from_int(n, width)
        classes[weight_trace(bits)].append(n)
    sizes = [len(v) for v in classes.values()]
    return classes, sizes

for width in [8, 10, 12, 14, 16]:
    classes, sizes = enumerate_weight_trace_classes(width)
    total = 1 << width
    unique = sum(1 for s in sizes if s == 1)
    max_size = max(sizes)
    mean_size = sum(sizes) / len(sizes)
    print(f"N={width:2d}: total={total:6d}, distinct_traces={len(classes):6d}, unique_classes={unique:6d}, max_class={max_size:4d}, mean_class={mean_size:.3f}")


# 11. Random Controls

Random inputs still create the same carrier geometry.

That proves the medium, not the message.

The source-specific question is:

$$
\text{Does the fold fingerprint separate sources from random controls?}
$$


In [ ]:
def random_binary_seed(N, rng=random):
    return [rng.randrange(2) for _ in range(N)]

def lock_count_for_seed(bits):
    rows = xor_rows(bits)
    trace = trace_metrics(rows)
    return len(trace["locks"])

rng = random.Random(12345)
trials = 100
random_lock_counts = [lock_count_for_seed(random_binary_seed(N, rng)) for _ in range(trials)]

pi_lock_count = len(binary_trace["locks"])
mu = np.mean(random_lock_counts)
sigma = np.std(random_lock_counts, ddof=1)

print("pi lock count:", pi_lock_count)
print("random controls:")
print("  mean =", mu)
print("  std  =", sigma)
print("  min  =", min(random_lock_counts))
print("  max  =", max(random_lock_counts))
print("  z(pi)=", (pi_lock_count - mu) / sigma if sigma > 0 else None)


# 12. Visual Verification


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0,0].plot(decimal_S, linewidth=1)
axes[0,0].axvline(13, linestyle="--", linewidth=1)
axes[0,0].axvline(448, linestyle="--", linewidth=1)
axes[0,0].axvline(512, linestyle="--", linewidth=1)
axes[0,0].set_title("Decimal Value-Channel Sum")
axes[0,0].set_xlabel("Level ℓ")
axes[0,0].set_ylabel("S_ℓ^(10)")

R = binary_trace["R"]
axes[0,1].plot(R, linewidth=0.7)
axes[0,1].scatter(binary_trace["locks"], [0]*len(binary_trace["locks"]), s=12)
axes[0,1].axhline(0, linewidth=1)
axes[0,1].set_title("Binary Residue Wave R_ℓ")
axes[0,1].set_xlabel("Level ℓ")
axes[0,1].set_ylabel("R_ℓ")

segment = R[20:1800]
axes[1,0].hist(segment, bins=50, edgecolor="black")
axes[1,0].axvline(0, linestyle="--", linewidth=1)
axes[1,0].set_title("Residue Distribution (ℓ=20..1800)")
axes[1,0].set_xlabel("R_ℓ")
axes[1,0].set_ylabel("Frequency")

last_levels = list(range(2032, 2048))
axes[1,1].bar([str(l) for l in last_levels], binary_trace["N"][last_levels], alpha=0.7, label="N")
axes[1,1].bar([str(l) for l in last_levels], binary_trace["S"][last_levels], alpha=0.7, label="S")
axes[1,1].set_title("Terminal Collapse: Last 16 Levels")
axes[1,1].set_xlabel("Level ℓ")
axes[1,1].set_ylabel("Value")
axes[1,1].tick_params(axis="x", rotation=60)
axes[1,1].legend()

plt.tight_layout()
plt.show()


In [ ]:
max_levels = 650
max_cols = 650

img = np.ones((max_levels, max_cols), dtype=float)
for level in range(max_levels):
    row = xor_lattice[level][:max_cols]
    for i, bit in enumerate(row):
        img[level, i] = 0.0 if bit else 1.0

plt.figure(figsize=(12, 10))
plt.imshow(img, cmap="gray", aspect="auto", interpolation="nearest")
for level in [110, 300, 448, 512]:
    plt.axhline(level, linestyle="--", linewidth=0.8)
    plt.text(max_cols + 5, level, f"ℓ={level}", va="center")
plt.title("Parity Shadow Lattice: Rule 90 / (I+E)^ℓ")
plt.xlabel("Position")
plt.ylabel("Level ℓ")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
axes = axes.ravel()

for idx, k in enumerate(range(0, 6)):
    level = N - (1 << k)
    row = xor_lattice[level]
    ax = axes[idx]
    ax.imshow(np.array([row]), cmap="gray_r", aspect="auto", interpolation="nearest")
    ax.set_title(f"k={k}, ℓ={level}, mod {1<<k}\\nrow length={len(row)}")
    ax.set_yticks([])
    ax.set_xlabel("Residue class")
    for i, bit in enumerate(row):
        ax.text(i, 0, str(bit), ha="center", va="center", fontsize=8)

plt.suptitle("Dyadic Terminal Checksum Cascade", fontsize=16)
plt.tight_layout()
plt.show()


# 13. Reverse Engine Skeleton

The practical full-size reverse engine proceeds:

1. Build linear constraints:

$$
Cx=y.
$$

2. Row-reduce over $GF(2)$.

3. Parameterize the affine space:

$$
x=x_0+Bz.
$$

4. Add nonlinear weight constraints:

$$
\operatorname{wt}(A_\ell(x_0+Bz))=S_\ell.
$$

5. Solve with SAT / MILP / branch-and-bound / custom parity-weight pruning.

This notebook verifies steps 1 and 2 for the full $2048$ system and demonstrates weight-trace rigidity for small $N$.


# 14. Shared Prompt for Other AIs

Use this to align Gemini, Kimi, Claude, or future ChatGPT sessions:

> We are working under the name **Nexus Fold Tomography** (`FOLD-TOMO`). The current notebook is **FOLD-TOMO-01: Trace-Sufficient Reverse Fold Engine**. The verified object is the first $2048$ fractional digits of $\pi$ under decimal adjacent-difference reduction. Its parity shadow is exact: $|a-b|\bmod2=a\oplus b$, so the binary fold is Rule 90, $x^{(\ell)}=(I+E)^\ell x^{(0)}$. Lucas's theorem gives $x_i^{(\ell)}=\bigoplus_{j\subseteq\ell}x_{i+j}^{(0)}$. The level $448$ mask is $\{0,64,128,192,256,320,384,448\}$ and the measured row is $N=1600,S=800,R=0$. Terminal rows $\ell=N-2^k$ are dyadic residue-class parity checks. Therefore the fold is a multiscale parity tomography machine. The reverse engine solves linear parity probes first, leaving a structured affine space, then applies nonlinear Hamming-weight constraints $\operatorname{wt}((I+E)^\ell x)=S_\ell$. Keep SHA and Collatz as branch-grammar analogies unless separately proven.


# 15. Final Lock

$$
\Delta:
\text{decimal digits project to parity}
$$

$$
\oplus:
\text{parity becomes Rule 90 / }(I+E)^\ell
$$

$$
↻:
\text{Lucas masks and dyadic terminal rows form tomography probes}
$$

$$
\bot:
\text{row-sum residues break remaining symmetry}
$$

$$
\Psi:
\boxed{
\text{shape can recover location when enough probes are stacked}
}
$$

The notebook name is fixed:

$$
\boxed{
\textbf{FOLD-TOMO-01: Trace-Sufficient Reverse Fold Engine}
}
$$
